# 01 — Embeddings Deep Dive

**Goal**: Understand embedding models, how they're trained, and how to choose the right one.

We'll use Ollama's embedding model and compare it with sentence-transformers.

In [ ]:
import requests
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

print("Imports OK")

## 1. Embedding Models: Options

| Model | Provider | Dim | Free? | Notes |
|---|---|---|---|---|
| `nomic-embed-text` | Ollama | 768 | Yes | Local, fast |
| `all-MiniLM-L6-v2` | HuggingFace | 384 | Yes | Fast, good for English |
| `bge-small-en-v1.5` | HuggingFace | 384 | Yes | Good retrieval |
| `text-embedding-3-small` | OpenAI | 1536 | No | Best quality (paid) |

For this course, we'll use Ollama's `nomic-embed-text` (fully local).

## 2. Ollama Embedding Function

In [ ]:
def get_embedding(text, model="nomic-embed-text"):
    """Get embedding from Ollama."""
    resp = requests.post("http://localhost:11434/api/embeddings", json={
        "model": model,
        "prompt": text
    })
    return np.array(resp.json()["embedding"])

# Test
emb = get_embedding("Hello world")
print(f"Dimension: {len(emb)}")
print(f"First 5 values: {emb[:5]}")

## 3. Comparing Embedding Models on Semantic Similarity

In [ ]:
# Load sentence-transformers for comparison
st_model = SentenceTransformer('all-MiniLM-L6-v2')

pairs = [
    ("I love machine learning", "Deep learning is fascinating"),  # similar
    ("I love machine learning", "The weather is nice today"),  # dissimilar
    ("The cat sat on the mat", "A dog is sleeping on the floor"),  # somewhat similar
]

print(f"{'Pair':<50} {'Ollama':<10} {'HF':<10}")
print("-" * 70)
for t1, t2 in pairs:
    # Ollama
    e1, e2 = get_embedding(t1), get_embedding(t2)
    ollama_sim = cosine_similarity([e1], [e2])[0][0]
    
    # Sentence Transformers
    e1_st, e2_st = st_model.encode([t1, t2])
    hf_sim = cosine_similarity([e1_st], [e2_st])[0][0]
    
    label = f"{t1[:25]}... ↔ {t2[:20]}..."
    print(f"{label:<50} {ollama_sim:<10.3f} {hf_sim:<10.3f}")

## 4. Batched Embeddings (For Performance)

Ollama supports batch embedding.

In [ ]:
texts = [
    "Transformers use self-attention",
    "RAG combines retrieval with generation",
    "Vector databases store embeddings",
    "Fine-tuning adapts models to tasks"
]

# Embed one by one (simple approach)
embeddings = np.array([get_embedding(t) for t in texts])
print(f"Batch shape: {embeddings.shape}")

# Compute similarity matrix
sim_matrix = cosine_similarity(embeddings)
print(f"\nSimilarity matrix:\n{np.round(sim_matrix, 2)}")

## 5. Normalization

Important: embeddings should be normalized before search.

In [ ]:
def normalize(vec):
    return vec / np.linalg.norm(vec)

# Demonstrate
raw = get_embedding("test")
normed = normalize(raw)

print(f"Raw vector norm: {np.linalg.norm(raw):.4f}")
print(f"Normalized norm: {np.linalg.norm(normed):.4f}")
print(f"Dot product of normalized = cosine similarity: {np.dot(normed, normed):.4f}")

## Key Takeaways

1. **Ollama embeddings** are free, local, and compatible with everything
2. **Different models** produce different similarity scores (pick one and stick with it)
3. **Batch embeddings** for performance
4. **Normalize** vectors for consistent cosine similarity
5. Embedding dimension matters: higher = more information but slower

**Next**: ChromaDB — store and search vectors efficiently →